In [15]:
import numpy as np
import pandas as pd
import tensorflow
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout,Subtract
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,accuracy_score
import matplotlib.pyplot as plt
import re
import nltk
import spacy
import seaborn  as sns
from tqdm import tqdm
from nltk.corpus import stopwords

In [17]:
import csv

In [69]:
with open("/mnt/d/DL-Algorithm/train_snli.txt",'r') as file:
    data = file.readlines()
    print("Data Loaded")
# now we will  create a CSV file for it
with open('data.csv','w') as csvfile:
    filename = ['source_txt','plagerism_txt','label']
    writer = csv.DictWriter(csvfile,fieldnames=filename)
    writer.writeheader()
    for line in tqdm(data):
        line = line.rstrip("\n")

        parts = line.split("\t")
        source_txt = parts[0]
        plagerism_txt = parts[1]
        label = (parts[2])
        writer.writerow({
            'source_txt':source_txt,
            'plagerism_txt':plagerism_txt,
            'label':label
        })
    print('csv file created successfully')
        


Data Loaded


100%|██████████| 367373/367373 [00:02<00:00, 125339.06it/s]

csv file created successfully


In [70]:
df = pd.read_csv("data.csv")

In [71]:
df.head(5)

,source_txt,plagerism_txt,label
0,A person on a horse jumps over a broken down a...,"A person is at a diner, ordering an omelette.",0
1,A person on a horse jumps over a broken down a...,"A person is outdoors, on a horse.",1
2,Children smiling and waving at camera,There are children present,1
3,Children smiling and waving at camera,The kids are frowning,0
4,A boy is jumping on skateboard in the middle o...,The boy skates down the sidewalk.,0


In [72]:
df.duplicated().sum()

np.int64(454)

In [73]:
df.drop_duplicates(inplace=True)

In [82]:
df.isna().sum()

source_txt       0
plagerism_txt    4
label            0
source_clean     0
dtype: int64

In [85]:
df=df.dropna()

In [86]:
df.isna().sum()

source_txt       0
plagerism_txt    0
label            0
source_clean     0
dtype: int64

In [87]:
import re,string
def clean_text(text):
    text  = text.lower()
    text = re.sub(r"\n"," ",text)
    text = text.translate(str.maketrans("","",string.punctuation))
    return text
df["source_clean"] = df['source_txt'].astype(str).apply(clean_text)
df['plag_clean']=df["plagerism_txt"].astype(str).apply(clean_text)


In [93]:
source = df['source_clean'].tolist()
plag = df['plag_clean'].tolist()
labels = df['label'].values
tokenizer = Tokenizer(num_words=20000,oov_token="<OOV>")
tokenizer.fit_on_texts(source+plag)

In [98]:
source_seq = tokenizer.texts_to_sequences(source)
plag_seq = tokenizer.texts_to_sequences(plag)
max_len = 50
source_pad =  pad_sequences(source_seq,maxlen=max_len)
plag_pad = pad_sequences(plag_seq,maxlen=max_len)

In [99]:
X_train_src,X_test_src,X_train_plag,X_test_plag,y_train,y_test = train_test_split(
    source_pad,plag_pad,labels,test_size=0.2,random_state=42
)

In [102]:
#Saimese LSTM network
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout, Subtract
from tensorflow.keras.models import Model
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 128
lstm_dim = 64

input_a = Input(shape=(max_len,))
input_b = Input(shape=(max_len,))

embedding = Embedding(vocab_size , embedding_dim , input_length = max_len)
lstm = LSTM(lstm_dim)

encoded_a = lstm(embedding(input_a))
encoded_b = lstm(embedding(input_b))

merged = Subtract()([encoded_a , encoded_b])
merged = Dense(64 , activation = "relu")(merged)
merged = Dropout(0.5)(merged)
out = Dense(1,activation="sigmoid")(merged)

model = Model(inputs=[input_a , input_b], outputs=out)
model.compile(loss = 'binary_crossentropy' , optimizer = 'adam' , metrics = ['accuracy'])
model.summary()


/mnt/d/DL-Algorithm/lvenv/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  trainable=True,
W0000 00:00:1790020607.397906    1131 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1790020607.412435    1131 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1790020607.844096    1131 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
I0000 00:00:1790020607.845677    1131 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5262 MB memory:  -> device: 0, name

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 50, 128)   │  3,874,432 │ input_layer[0][0… │
│ (Embedding)         │                   │            │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 64)        │     49,408 │ embedding[0][0],  │
│                     │                   │            │ embedding[1][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ subtract (Subtract) │ (None, 64)        │          0 │ lstm[0][0],       │
│                     │                   │            │ lstm[1][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      4,160 │ subtract[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │         65 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,928,065 (14.98 MB)

 Trainable params: 3,928,065 (14.98 MB)

 Non-trainable params: 0 (0.00 B)

In [103]:
from tensorflow.keras.callbacks import EarlyStopping

es = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)


In [104]:

history = model.fit(
    [X_train_src, X_train_plag],
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=32,
    callbacks=[es],
    verbose=1
)

Epoch 1/10


I0000 00:00:1790020668.404773   10556 cuda_dnn.cc:461] Loaded cuDNN version 92400


7339/7339 ━━━━━━━━━━━━━━━━━━━━ 167s 16ms/step - accuracy: 0.7862 - loss: 0.4428 - val_accuracy: 0.8317 - val_loss: 0.3698
Epoch 2/10
7339/7339 ━━━━━━━━━━━━━━━━━━━━ 122s 17ms/step - accuracy: 0.8544 - loss: 0.3388 - val_accuracy: 0.8583 - val_loss: 0.3341
Epoch 3/10
7339/7339 ━━━━━━━━━━━━━━━━━━━━ 122s 17ms/step - accuracy: 0.8825 - loss: 0.2866 - val_accuracy: 0.8634 - val_loss: 0.3311
Epoch 4/10
7339/7339 ━━━━━━━━━━━━━━━━━━━━ 124s 17ms/step - accuracy: 0.8997 - loss: 0.2513 - val_accuracy: 0.8646 - val_loss: 0.3360
Epoch 5/10
7339/7339 ━━━━━━━━━━━━━━━━━━━━ 124s 17ms/step - accuracy: 0.9126 - loss: 0.2220 - val_accuracy: 0.8641 - val_loss: 0.3587


In [105]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [106]:
model.save("siamese_model.keras")